# Entraînement SO-101 (ACT)

Le dataset et le nom du modèle ont été choisis dans l’interface du robot.

1. **Exécution → Tout exécuter** (Ctrl+F9).
2. Coller le token Hugging Face quand il est demandé.
3. Laisser cet onglet ouvert jusqu’à la fin : le modèle est alors publié sur Hugging Face.

In [ ]:
!pip install -q lerobot==0.4.4
!pip install -q "transformers<5"

In [ ]:
# Connexion à Hugging Face : secret Colab « HF_TOKEN » s’il existe, sinon le token est demandé ici.
import json, re
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download, login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
login(token=token or getpass("Token Hugging Face (droit d’écriture) : "), add_to_git_credential=False)
user = HfApi().whoami()["name"]
try:
    path = hf_hub_download(f"{user}/so101-colab", "config.json", repo_type="dataset", force_download=True)
except Exception:
    raise SystemExit(f"Aucun entraînement préparé pour le compte Hugging Face « {user} ».\n"
                     "1. Dans l’interface du robot, cliquer sur « Entraîner sur Google Colab », puis « Préparer et ouvrir Colab ».\n"
                     "2. Utiliser ici un token du même compte que celui connecté sur le Raspberry, avec le droit d’écriture.\n"
                     "Puis relancer : Exécution → Tout exécuter.") from None
cfg = json.load(open(path, encoding="utf-8"))
REPO = re.compile(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,95}/[A-Za-z0-9][A-Za-z0-9_.-]{0,95}")
assert REPO.fullmatch(cfg["dataset_repo"]) and REPO.fullmatch(cfg["model_repo"]), "Réglages invalides"
assert re.fullmatch(r"[A-Za-z0-9_-]{1,55}", cfg["job_name"]), "Nom de modèle invalide"
DATASET, MODEL, JOB = cfg["dataset_repo"], cfg["model_repo"], cfg["job_name"]
STEPS, SAVE = int(cfg["steps"]), int(cfg["save_freq"])
print(f"Dataset : {DATASET}\nModèle publié sur : {MODEL}\nÉtapes : {STEPS}")

## Entraînement
Durée indicative : plusieurs heures pour une vingtaine d’épisodes.

In [ ]:
!lerobot-train \
  --dataset.repo_id={DATASET} \
  --policy.type=act \
  --policy.device=cuda \
  --policy.repo_id={MODEL} \
  --output_dir=outputs/train/{JOB} \
  --job_name={JOB} \
  --steps={STEPS} \
  --save_freq={SAVE} \
  --wandb.enable=false

## Reprise après une coupure
Mettre `REPRENDRE = True`, puis exécuter seulement cette cellule.

In [ ]:
REPRENDRE = False  # passer à True seulement si la session Colab s’est coupée pendant l’entraînement
if REPRENDRE:
    !lerobot-train --config_path=outputs/train/{JOB}/checkpoints/last/pretrained_model/train_config.json --resume=true